# Vietnamese Sentiment Analysis with PhoBERT and FastText

Notebook nay gom toan bo pipeline don gian trong 1 file:

- Cai dat thu vien
- Tien xu ly du lieu
- Train PhoBERT
- Danh gia PhoBERT
- Train FastText
- Danh gia FastText
- Du doan nhanh voi ca 2 mo hinh

Nhan:

- `0`: negative
- `1`: positive
- `2`: neutral

In [ ]:
# Neu chua cai thu vien thi bo comment dong duoi va chay 1 lan
# !pip install pandas numpy scikit-learn underthesea torch transformers datasets accelerate evaluate safetensors fasttext-wheel

In [ ]:
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)
from underthesea import word_tokenize

try:
    import fasttext
except ImportError:
    fasttext = None
    print("FastText chua duoc cai. Hay chay cell pip install neu ban muon train FastText.")

In [ ]:
SEED = 42
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5

FASTTEXT_EPOCHS = 25
FASTTEXT_LR = 0.5
FASTTEXT_WORD_NGRAMS = 2
FASTTEXT_DIM = 100

RAW_TRAIN_PATH = "data/preprocessed/train.csv"
RAW_TEST_PATH = "data/preprocessed/test.csv"
PROCESSED_DIR = Path("data/processed")
PHOBERT_OUTPUT_DIR = Path("outputs/phobert-sentiment")
FASTTEXT_OUTPUT_DIR = Path("outputs/fasttext")

LABEL_ID_TO_NAME = {0: "negative", 1: "positive", 2: "neutral"}
LABEL_NAME_TO_ID = {value: key for key, value in LABEL_ID_TO_NAME.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PHOBERT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FASTTEXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 1. Load resources va dinh nghia ham preprocess

In [ ]:
with open("resources/teencode.json", "r", encoding="utf-8") as file:
    teencode_dict = json.load(file)

with open("resources/stopword.json", "r", encoding="utf-8") as file:
    stopword_data = json.load(file)

final_stopwords = set(stopword_data["remove_always"]) - set(stopword_data["keep_for_sentiment"])


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [teencode_dict.get(word, word) for word in words]

    text = " ".join(words)
    text = word_tokenize(text, format="text")

    words = [word for word in text.split() if word not in final_stopwords]
    return " ".join(words)


def save_fasttext_format(df: pd.DataFrame, output_path: Path) -> None:
    lines = "__label__" + df["flag"].astype(str) + " " + df["comments_clean"].astype(str)
    lines.to_csv(output_path, index=False, header=False, encoding="utf-8")

## 2. Tien xu ly va chia train/validation

In [ ]:
train_full_df = pd.read_csv(RAW_TRAIN_PATH)
test_df = pd.read_csv(RAW_TEST_PATH)

train_full_df["comments_clean"] = train_full_df["comments"].apply(clean_text)
test_df["comments_clean"] = test_df["comments"].apply(clean_text)

train_full_df = train_full_df.dropna(subset=["comments_clean"])
train_full_df = train_full_df[train_full_df["comments_clean"].str.strip() != ""].copy()
test_df = test_df.dropna(subset=["comments_clean"])
test_df = test_df[test_df["comments_clean"].str.strip() != ""].copy()

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_full_df["flag"],
)

train_df.to_csv(PROCESSED_DIR / "train_new_cleaned.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(PROCESSED_DIR / "val_cleaned.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(PROCESSED_DIR / "test_cleaned.csv", index=False, encoding="utf-8-sig")

save_fasttext_format(train_df, PROCESSED_DIR / "train_fasttext.txt")
save_fasttext_format(val_df, PROCESSED_DIR / "val_fasttext.txt")
save_fasttext_format(test_df, PROCESSED_DIR / "test_fasttext.txt")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("\nLabel distribution train:")
print(train_df["flag"].value_counts().sort_index())

In [ ]:
train_df[["comments", "comments_clean", "flag"]].head(10)

## 3. Chuyen sang Hugging Face Dataset cho PhoBERT

In [ ]:
def build_dataset(df: pd.DataFrame, text_column: str = "comments_clean", label_column: str = "flag") -> Dataset:
    df = df[[text_column, label_column]].dropna().copy()
    df[text_column] = df[text_column].astype(str).str.strip()
    df = df[df[text_column] != ""]
    df[label_column] = df[label_column].astype(int)
    dataset = Dataset.from_pandas(df, preserve_index=False)
    return dataset.rename_column(label_column, "label")


train_dataset = build_dataset(train_df)
val_dataset = build_dataset(val_df)
test_dataset = build_dataset(test_df)

train_dataset

## 4. Tokenize voi PhoBERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(
        batch["comments_clean"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


train_tokenized = train_dataset.map(tokenize_batch, batched=True)
val_tokenized = val_dataset.map(tokenize_batch, batched=True)
test_tokenized = test_dataset.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 5. Train PhoBERT

In [ ]:
phobert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=LABEL_ID_TO_NAME,
    label2id=LABEL_NAME_TO_ID,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir=str(PHOBERT_OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)


phobert_trainer = Trainer(
    model=phobert_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

phobert_trainer.train()

In [ ]:
phobert_val_metrics = phobert_trainer.evaluate()
phobert_val_metrics

In [ ]:
phobert_trainer.save_model(str(PHOBERT_OUTPUT_DIR))
tokenizer.save_pretrained(str(PHOBERT_OUTPUT_DIR))
print("Saved PhoBERT model to", PHOBERT_OUTPUT_DIR)

## 6. Danh gia PhoBERT tren tap test

In [ ]:
phobert_test_predictions = phobert_trainer.predict(test_tokenized)
phobert_test_pred_labels = np.argmax(phobert_test_predictions.predictions, axis=1)
phobert_test_true_labels = np.array(test_tokenized["label"])

print("PhoBERT Accuracy:", round(accuracy_score(phobert_test_true_labels, phobert_test_pred_labels), 4))
print("PhoBERT Macro F1:", round(f1_score(phobert_test_true_labels, phobert_test_pred_labels, average="macro"), 4))
print("PhoBERT Weighted F1:", round(f1_score(phobert_test_true_labels, phobert_test_pred_labels, average="weighted"), 4))
print("\nPhoBERT classification report:\n")
print(classification_report(phobert_test_true_labels, phobert_test_pred_labels, digits=4))

## 7. Du doan nhanh voi PhoBERT

In [ ]:
phobert_classifier = pipeline(
    task="text-classification",
    model=str(PHOBERT_OUTPUT_DIR),
    tokenizer=str(PHOBERT_OUTPUT_DIR),
    truncation=True,
)


def predict_phobert_sentiment(raw_text: str):
    cleaned = clean_text(raw_text)
    result = phobert_classifier(cleaned)[0]
    return {
        "model": "PhoBERT",
        "raw_text": raw_text,
        "cleaned_text": cleaned,
        "label": result["label"],
        "score": round(result["score"], 4),
    }


predict_phobert_sentiment("quan an nay kha on, phuc vu nhanh va lich su")

## 8. Train FastText

In [ ]:
if fasttext is None:
    raise ImportError("Hay cai fasttext-wheel truoc khi chay phan FastText.")

fasttext_model = fasttext.train_supervised(
    input=str(PROCESSED_DIR / "train_fasttext.txt"),
    lr=FASTTEXT_LR,
    epoch=FASTTEXT_EPOCHS,
    wordNgrams=FASTTEXT_WORD_NGRAMS,
    dim=FASTTEXT_DIM,
    loss="softmax",
    seed=SEED,
)

fasttext_model.save_model(str(FASTTEXT_OUTPUT_DIR / "fasttext_sentiment.bin"))
print("Saved FastText model to", FASTTEXT_OUTPUT_DIR / "fasttext_sentiment.bin")

In [ ]:
fasttext_model.test(str(PROCESSED_DIR / "val_fasttext.txt"))

## 9. Danh gia FastText tren tap test

In [ ]:
def parse_fasttext_label(label: str) -> int:
    return int(label.replace("__label__", ""))


fasttext_test_df = test_df[["comments_clean", "flag"]].copy()
fasttext_predictions = [parse_fasttext_label(fasttext_model.predict(text)[0][0]) for text in fasttext_test_df["comments_clean"]]
fasttext_true_labels = fasttext_test_df["flag"].astype(int).tolist()

print("FastText Accuracy:", round(accuracy_score(fasttext_true_labels, fasttext_predictions), 4))
print("FastText Macro F1:", round(f1_score(fasttext_true_labels, fasttext_predictions, average="macro"), 4))
print("FastText Weighted F1:", round(f1_score(fasttext_true_labels, fasttext_predictions, average="weighted"), 4))
print("\nFastText classification report:\n")
print(classification_report(fasttext_true_labels, fasttext_predictions, digits=4))

## 10. Du doan nhanh voi FastText

In [ ]:
def predict_fasttext_sentiment(raw_text: str):
    cleaned = clean_text(raw_text)
    label, score = fasttext_model.predict(cleaned)
    label_id = parse_fasttext_label(label[0])
    return {
        "model": "FastText",
        "raw_text": raw_text,
        "cleaned_text": cleaned,
        "label": LABEL_ID_TO_NAME[label_id],
        "score": round(float(score[0]), 4),
    }


predict_fasttext_sentiment("quan an nay kha on, phuc vu nhanh va lich su")

## 11. So sanh nhanh PhoBERT va FastText

In [ ]:
examples = [
    "mon nay ngon, minh se quay lai",
    "phuc vu lau va do an qua te",
    "quan dong, tam on, khong co gi dac biet",
]

comparison_rows = []
for text in examples:
    comparison_rows.append(predict_phobert_sentiment(text))
    comparison_rows.append(predict_fasttext_sentiment(text))

pd.DataFrame(comparison_rows)

## 12. Ghi chu de demo do an

- Neu may yeu, giam `BATCH_SIZE` xuong `4` hoac `2` cho PhoBERT.
- FastText train nhanh hon nhieu, rat hop de demo hoac lam baseline.
- PhoBERT thuong cho ket qua tot hon nhung ton tai nguyen hon.
- Notebook nay phu hop de nop/bao cao vi ca 2 huong nam trong 1 file.